# Agents example 1: Our first StateGraph

Run these cells from top to bottom in Google Colab. This first step installs LangGraph and checks its imports. No API key, GPU, or external service connection is required. The installation downloads packages from PyPI.

LangGraph installs its required dependencies automatically. We will add a model provider package later if the example needs one.

In [ ]:
# Install LangGraph in the current Colab runtime.
%pip install -q "langgraph>=1.0,<2.0"

## Check that installation worked

`StateGraph` lets us define a graph. `START` and `END` mark its beginning and end. `TypedDict` describes the state passed between nodes and comes with Python.

This cell only checks imports; we will build the graph next.

In [ ]:
from importlib.metadata import version
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph version:", version("langgraph"))
print("Setup complete! Ready to build our first graph.")

## Colab runtime notes

Run the installation cell again whenever Colab starts a fresh runtime. If Colab asks you to restart the runtime after installation, restart it and rerun the import cell.

[Official installation guide](https://docs.langchain.com/oss/python/langgraph/install)

## 1. What is a StateGraph?

A **StateGraph** is a workflow whose steps share data called **state**.

- **State:** a dictionary containing the current data.
- **Node:** a Python function that reads state and returns updates.
- **Edge:** a connection that determines which node runs next.

We will start with the number 4, add 2, then multiply by 3:

```text
START -> add_two -> multiply_by_three -> END
   4         6               18
```

START and END are special markers, not functions we write. This example uses ordinary Python arithmetic: no model or API calls.


## 2. Describe the state

Our shared dictionary has just one field, `number`. `TypedDict` describes its expected structure for readers and type checkers; it does not validate values at runtime.


In [ ]:
class NumberState(TypedDict):
    number: int


## 3. Define two nodes

Each node receives the current state and returns a dictionary containing its updates. For this simple field, LangGraph replaces the old number with the returned number. We do not change the input dictionary directly.

Defining these functions does not run them yet.


In [ ]:
def add_two(state: NumberState):
    return {"number": state["number"] + 2}


def multiply_by_three(state: NumberState):
    return {"number": state["number"] * 3}


## 4. Build and compile the graph

`add_node` gives each function a name in the graph. `add_edge` connects the steps in order. `compile()` turns our definition into a runnable graph; it does not execute it yet.


In [ ]:
builder = StateGraph(NumberState)

builder.add_node("add_two", add_two)
builder.add_node("multiply_by_three", multiply_by_three)

builder.add_edge(START, "add_two")
builder.add_edge("add_two", "multiply_by_three")
builder.add_edge("multiply_by_three", END)

graph = builder.compile()


## 5. Run the graph

`invoke()` takes the initial state, executes the connected nodes, and returns the final state. The answer is `(4 + 2) * 3 = 18`.


In [ ]:
initial_state = {"number": 4}
final_state = graph.invoke(initial_state)
print("Initial state:", initial_state)
print("Final state:", final_state)

assert final_state == {"number": 18}


Expected output:

```text
Initial state: {'number': 4}
Final state: {'number': 18}
```

## 6. See the updates from each node

This starts a new run from 4. `stream_mode="updates"` shows the changes returned by each node as it finishes.


In [ ]:
for update in graph.stream({"number": 4}, stream_mode="updates"):
    print(update)


Expected output:

```text
{'add_two': {'number': 6}}
{'multiply_by_three': {'number': 18}}
```

## Try it yourself

Change `starting_number` below and predict the result before running. For 10, the result should be 36. Each invocation starts with the state you supply; it does not reuse the previous result.


In [ ]:
starting_number = 10
print(graph.invoke({"number": starting_number}))


The node functions define **what happens**, edges define **the order**, and state carries **the data**. This graph has one fixed path. Later, graphs can also include branches and loops.

[Official StateGraph reference](https://reference.langchain.com/python/langgraph/graph/state/StateGraph)
